# Notebook 1 — IDRiD Segmentation Preprocessing
**Paper:** *A Comprehensive Deep-Learning Framework Integrating Lesion Segmentation and Stage Classification for Enhanced Diabetic Retinopathy Diagnosis*  
**Authors:** Incir & Bozkurt | Int J Imaging Syst Tech, 2026

---

## What this notebook does
Preprocesses the IDRiD segmentation dataset (81 high-resolution fundus images) and generates
training-ready patches for four lesion types:
**EX** (Hard Exudates), **HE** (Haemorrhages), **MA** (Microaneurysms), **SE** (Soft Exudates).

## Pipeline (Section 3.3 of the paper)
1. Mount Google Drive + install dependencies  
2. Extract `A. Segmentation.zip` to Colab local SSD (faster than Drive I/O)  
3. Build image index → **80 / 10 / 10** image-level split  
4. Remove optic disc using ground-truth OD masks  
5. Resize → 4096x2560 → patch 1024x1024 (50% overlap) → filter empty → resize 224x224  
6. Augment training patches x3 (rotation, noise, sharpening, flip, crop)  
7. Save compressed `.npz` files to Drive  

## Expected I/O
| | Path |
|---|---|
| **Input** | `MyDrive/DR_PROJECT/datasets/A. Segmentation.zip` |
| **Output** | `MyDrive/DR_PROJECT/preprocessed/segmentation/{EX,HE,MA,SE}/{train,val,test}.npz` |

> **Note:** GPU is not required for this notebook. CPU is sufficient.

## Cell 1 — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm

import os, random, zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

print('Libraries loaded.')

## Cell 2 — Configuration
All paths and paper-specified preprocessing parameters in one place.  
Change `BASE_PATH` if your Google Drive folder structure differs.

In [ ]:
BASE_PATH     = '/content/drive/MyDrive/DR_PROJECT/datasets'
OUTPUT_BASE   = '/content/drive/MyDrive/DR_PROJECT/preprocessed'
IDRID_SEG_ZIP = os.path.join(BASE_PATH, 'A. Segmentation.zip')

EXTRACT_DIR = '/content/idrid_seg'  # Colab local SSD — fast I/O during processing
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_BASE, exist_ok=True)

# ---- Paper preprocessing parameters (Section 3.3) ----
RESIZE_W, RESIZE_H = 4096, 2560  # resize before patching
PATCH_SIZE  = 1024
STRIDE      = 512                 # 50% overlap -> 28 patches per image (max)
FINAL_SIZE  = 224                 # CNN input size
AUGMENT_FACTOR = 3                # 3 new variants per original patch (training only)
RANDOM_SEED = 42

LESIONS = ['EX', 'HE', 'MA', 'SE']

LESION_FOLDER_MAP = {
    'EX': '3. Hard Exudates',
    'HE': '2. Haemorrhages',
    'MA': '1. Microaneurysms',
    'SE': '4. Soft Exudates',
}
LESION_SUFFIX_MAP = {'EX': '_EX', 'HE': '_HE', 'MA': '_MA', 'SE': '_SE'}

print(f'Resize  : {RESIZE_W} x {RESIZE_H}')
print(f'Patch   : {PATCH_SIZE} x {PATCH_SIZE}  stride {STRIDE}  (50% overlap)')
print(f'Final   : {FINAL_SIZE} x {FINAL_SIZE}')
print(f'Augment : x{AUGMENT_FACTOR} on training patches only')

## Cell 3 — Extract Zip & Locate Dataset
We extract to Colab's local SSD (`/content/`) rather than working directly on Google Drive.  
Drive I/O on thousands of small image files is ~10x slower than the local SSD.

In [ ]:
print('Extracting zip to local SSD (may take ~1 min)...')
with zipfile.ZipFile(IDRID_SEG_ZIP, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Done.')

# Auto-detect dataset root regardless of zip nesting depth
SEG_BASE = None
for root, dirs, files in os.walk(EXTRACT_DIR):
    if '1. Original Images' in dirs and '2. All Segmentation Groundtruths' in dirs:
        SEG_BASE = root
        break

if SEG_BASE is None:
    raise FileNotFoundError('Dataset root not found in zip. Check zip structure.')

print(f'Dataset root found: {SEG_BASE}')

TRAIN_IMG_DIR  = os.path.join(SEG_BASE, '1. Original Images', 'a. Training Set')
TEST_IMG_DIR   = os.path.join(SEG_BASE, '1. Original Images', 'b. Testing Set')
TRAIN_MASK_DIR = os.path.join(SEG_BASE, '2. All Segmentation Groundtruths', 'a. Training Set')
TEST_MASK_DIR  = os.path.join(SEG_BASE, '2. All Segmentation Groundtruths', 'b. Testing Set')
TRAIN_OD_DIR   = os.path.join(TRAIN_MASK_DIR, '5. Optic Disc')
TEST_OD_DIR    = os.path.join(TEST_MASK_DIR,  '5. Optic Disc')

print(f'\nTrain images  : {len(os.listdir(TRAIN_IMG_DIR))}')
print(f'Test  images  : {len(os.listdir(TEST_IMG_DIR))}')
print(f'Train OD masks: {len(os.listdir(TRAIN_OD_DIR))}')
for lesion in LESIONS:
    path = os.path.join(TRAIN_MASK_DIR, LESION_FOLDER_MAP[lesion])
    print(f'Train {lesion} masks : {len(os.listdir(path))}')

## Cell 4 — Build Image Index & 80/10/10 Split

The IDRiD dataset ships with an official train (54) / test (27) split.  
The paper recombines all 81 images and re-splits: **80% train / 10% val / 10% test**.

> **Why split at image level?**  
> Patches from the same fundus image must never appear in both train and test sets.  
> Patch-level splitting causes the model to see the same patient twice, inflating test metrics artificially.

In [ ]:
all_images = []

def _parse_entry(fname, img_dir, od_dir, mask_root):
    if not fname.lower().endswith('.jpg'):
        return None
    img_id = fname.split('_')[1].split('.')[0]  # 'IDRiD_01.jpg' -> '01'
    return {
        'fname':     fname,
        'img_id':    img_id,
        'img_path':  os.path.join(img_dir,  fname),
        'od_path':   os.path.join(od_dir,   f'IDRiD_{img_id}_OD.tif'),
        'mask_root': mask_root,
    }

for fname in sorted(os.listdir(TRAIN_IMG_DIR)):
    entry = _parse_entry(fname, TRAIN_IMG_DIR, TRAIN_OD_DIR, TRAIN_MASK_DIR)
    if entry: all_images.append(entry)

for fname in sorted(os.listdir(TEST_IMG_DIR)):
    entry = _parse_entry(fname, TEST_IMG_DIR, TEST_OD_DIR, TEST_MASK_DIR)
    if entry: all_images.append(entry)

print(f'Total images indexed: {len(all_images)}')

# Shuffle image indices, then assign splits sequentially
random.seed(RANDOM_SEED)
indices = list(range(len(all_images)))
random.shuffle(indices)

n       = len(indices)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

split_map = {}
for rank, idx in enumerate(indices):
    if rank < n_train:
        split_map[idx] = 'train'
    elif rank < n_train + n_val:
        split_map[idx] = 'val'
    else:
        split_map[idx] = 'test'

for idx, info in enumerate(all_images):
    info['split'] = split_map[idx]

counts = {s: sum(1 for i in all_images if i['split'] == s) for s in ['train', 'val', 'test']}
print(f'Image split -> train: {counts["train"]}, val: {counts["val"]}, test: {counts["test"]}')

print('\nSample assignments (first 6 images):')
for info in all_images[:6]:
    print(f'  {info["fname"]}  ->  {info["split"]}')

## Cell 5 — Preprocessing & Augmentation Functions

| Function | Purpose |
|---|---|
| `load_rgb` | Load image as RGB (OpenCV loads as BGR by default) |
| `remove_optic_disc` | Inpaint OD region using the ground-truth OD mask |
| `extract_patches` | Slide 1024x1024 window; keep only patches with lesion content |
| `to_final` | Resize patch to 224x224 (nearest-neighbor for mask to keep binary values) |
| `aug_rotate_noise` | Variant 1: random rotation + Gaussian noise |
| `aug_sharpen_flip` | Variant 2: unsharp masking + horizontal flip |
| `aug_crop_rotate` | Variant 3: random 80–95% crop + rotation |

In [ ]:
def load_rgb(path):
    img = cv2.imread(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None


def remove_optic_disc(image, od_mask):
    '''Replace OD region with inpainted content (Section 3.3).
    The OD is a bright non-pathological structure that mimics hard exudates
    and causes false-positive activations near its boundary.
    We use the ground-truth OD masks from IDRiD (folder 5. Optic Disc)
    rather than Hough-circle detection for maximum accuracy.
    '''
    binary  = (od_mask > 127).astype(np.uint8) * 255
    kernel  = np.ones((20, 20), np.uint8)
    dilated = cv2.dilate(binary, kernel, iterations=2)  # slight over-dilation for safety
    return cv2.inpaint(image, dilated, inpaintRadius=7, flags=cv2.INPAINT_TELEA)


def extract_patches(image, mask):
    '''Sliding 1024x1024 window with stride=512 (50% overlap).
    Only keeps patches where the lesion mask contains at least one positive pixel.
    This removes background-only patches which would bias training toward
    predicting "no lesion".
    '''
    h, w = image.shape[:2]
    patches = []
    for y in range(0, h - PATCH_SIZE + 1, STRIDE):
        for x in range(0, w - PATCH_SIZE + 1, STRIDE):
            ip = image[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            mp = mask[y:y+PATCH_SIZE,  x:x+PATCH_SIZE]
            if mp.max() > 0:
                patches.append((ip, mp))
    return patches


def to_final(img, msk):
    '''Resize to 224x224.
    INTER_LINEAR for images (smooth downsampling).
    INTER_NEAREST for masks (preserves binary 0/255 values without blending).
    '''
    img_r = cv2.resize(img, (FINAL_SIZE, FINAL_SIZE), interpolation=cv2.INTER_LINEAR)
    msk_r = cv2.resize(msk, (FINAL_SIZE, FINAL_SIZE), interpolation=cv2.INTER_NEAREST)
    return img_r, (msk_r > 127).astype(np.uint8) * 255


def _rot(img, angle, is_mask=False):
    '''Helper: rotate image or mask around center. BORDER_REFLECT avoids black borders.'''
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    return cv2.warpAffine(img, M, (w, h), flags=interp, borderMode=cv2.BORDER_REFLECT)


def aug_rotate_noise(img, msk):
    '''Augmentation variant 1: random rotation + Gaussian noise.
    Rotation teaches the model that lesions can appear at any orientation.
    Noise simulates image acquisition variability.
    '''
    angle = random.choice([15, 30, 45, 90, 135, 180, 270])
    img   = _rot(img, angle)
    msk   = _rot(msk, angle, is_mask=True)
    noise = np.random.normal(0, 8, img.shape).astype(np.int16)
    img   = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return img, msk


def aug_sharpen_flip(img, msk):
    '''Augmentation variant 2: unsharp masking + horizontal flip.
    Sharpening makes lesion boundaries more distinct.
    Flipping doubles spatial diversity (fundus images have no fixed left/right orientation).
    '''
    blurred   = cv2.GaussianBlur(img, (0, 0), 3)
    sharpened = cv2.addWeighted(img, 1.5, blurred, -0.5, 0)
    sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
    return cv2.flip(sharpened, 1), cv2.flip(msk, 1)


def aug_crop_rotate(img, msk):
    '''Augmentation variant 3: random rotation + crop to 80-95% of size.
    Cropping forces the model to handle partial lesion views,
    improving robustness to lesions at patch boundaries.
    '''
    angle = random.choice([15, 45, 90, 135])
    img   = _rot(img, angle)
    msk   = _rot(msk, angle, is_mask=True)
    h, w  = img.shape[:2]
    scale = random.uniform(0.80, 0.95)
    nh, nw = int(h * scale), int(w * scale)
    y0, x0 = random.randint(0, h - nh), random.randint(0, w - nw)
    return img[y0:y0+nh, x0:x0+nw], msk[y0:y0+nh, x0:x0+nw]


AUG_FNS = [aug_rotate_noise, aug_sharpen_flip, aug_crop_rotate]

print('All preprocessing functions defined.')

## Cell 6 — Run Preprocessing
Processes one lesion type at a time. Expected duration: **~20–35 min total** (CPU only, no GPU needed).

**If Colab disconnects:** re-run from Cell 1. The function checks for existing `.npz` files
and skips any lesion that has already been fully processed.

> **Expected SE messages:** SE (Soft Exudates) is only annotated in ~40 of the 81 IDRiD images.  
> `continue` (skip) messages for the other ~41 images are **normal and expected**.

In [ ]:
def preprocess_lesion(lesion):
    out_dir = os.path.join(OUTPUT_BASE, 'segmentation', lesion)
    os.makedirs(out_dir, exist_ok=True)

    # Idempotent: skip if all three split files already exist on Drive
    if all(os.path.exists(os.path.join(out_dir, f'{s}.npz')) for s in ['train', 'val', 'test']):
        print(f'[{lesion}] Already preprocessed - skipping.')
        return

    buf = {s: {'images': [], 'masks': []} for s in ['train', 'val', 'test']}

    for info in tqdm(all_images, desc=f'[{lesion}]'):
        mask_path = os.path.join(
            info['mask_root'],
            LESION_FOLDER_MAP[lesion],
            f"IDRiD_{info['img_id']}{LESION_SUFFIX_MAP[lesion]}.tif"
        )
        if not os.path.exists(mask_path):
            continue  # SE only annotated in ~40/81 images - expected

        image = load_rgb(info['img_path'])
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        od    = cv2.imread(info['od_path'], cv2.IMREAD_GRAYSCALE)

        if image is None or mask is None:
            print(f"  Warning: failed to load {info['fname']}")
            continue

        # Step 1: Remove optic disc using ground-truth OD mask
        if od is not None:
            image = remove_optic_disc(image, od)

        # Step 2: Resize to 4096x2560 (paper Section 3.3)
        image = cv2.resize(image, (RESIZE_W, RESIZE_H), interpolation=cv2.INTER_LINEAR)
        mask  = cv2.resize(mask,  (RESIZE_W, RESIZE_H), interpolation=cv2.INTER_NEAREST)
        mask  = (mask > 127).astype(np.uint8) * 255

        # Step 3: Extract 1024x1024 patches; discard all-black masks
        patches = extract_patches(image, mask)
        if not patches:
            continue

        split = info['split']
        for ip, mp in patches:
            # Step 4: Resize each valid patch to 224x224
            img_f, msk_f = to_final(ip, mp)
            buf[split]['images'].append(img_f)
            buf[split]['masks'].append(msk_f)

            # Step 5: Apply 3 augmentation variants to training patches only.
            # Augmentation is applied at 1024px scale BEFORE resizing to 224px
            # so spatial distortions are correctly scaled down.
            if split == 'train':
                for aug_fn in AUG_FNS:
                    ai, am = aug_fn(ip, mp)
                    ai, am = to_final(ai, am)
                    buf[split]['images'].append(ai)
                    buf[split]['masks'].append(am)

    # Save as compressed numpy archives to Drive
    # .npz is much faster to load during training than reading thousands of PNG files
    for split, data in buf.items():
        if not data['images']:
            print(f'  WARNING: 0 patches for {lesion}/{split}')
            continue
        imgs = np.stack(data['images']).astype(np.uint8)  # (N, 224, 224, 3)
        msks = np.stack(data['masks']).astype(np.uint8)   # (N, 224, 224)
        path = os.path.join(out_dir, f'{split}.npz')
        np.savez_compressed(path, images=imgs, masks=msks)
        print(f'  [{lesion}/{split}] {imgs.shape[0]} patches saved -> {path}')


for lesion in LESIONS:
    print(f"\n{'='*50}\nLesion: {lesion}\n{'='*50}")
    preprocess_lesion(lesion)

print('\nAll lesions preprocessed!')

## Cell 7 — Verify Outputs & Visualize Samples
Prints a patch-count summary table and shows sample image/mask pairs per lesion type.  
Confirm the patch counts are reasonable before moving to Notebook 2.

In [ ]:
# ---- Summary table ----
print(f"\n{'Lesion':<6} {'Train':>8} {'Val':>6} {'Test':>6}  {'Total':>7}")
print('-' * 38)
for lesion in LESIONS:
    out_dir = os.path.join(OUTPUT_BASE, 'segmentation', lesion)
    row = {}
    for split in ['train', 'val', 'test']:
        p = os.path.join(out_dir, f'{split}.npz')
        row[split] = int(np.load(p)['images'].shape[0]) if os.path.exists(p) else 0
    total = sum(row.values())
    print(f"{lesion:<6} {row['train']:>8} {row['val']:>6} {row['test']:>6}  {total:>7}")

# ---- Sample visualisation ----
fig, axes = plt.subplots(len(LESIONS), 7, figsize=(21, 4 * len(LESIONS)))

for r, lesion in enumerate(LESIONS):
    p = os.path.join(OUTPUT_BASE, 'segmentation', lesion, 'train.npz')
    if not os.path.exists(p):
        continue
    d = np.load(p)
    imgs, msks = d['images'], d['masks']
    has_lesion = msks.max(axis=(1, 2)) > 0
    sample_idx = np.where(has_lesion)[0][:3]

    axes[r, 0].text(0.5, 0.5, lesion, ha='center', va='center',
                    fontsize=16, fontweight='bold')
    axes[r, 0].axis('off')

    for c, i in enumerate(sample_idx):
        axes[r, c * 2 + 1].imshow(imgs[i])
        axes[r, c * 2 + 1].set_title('Image patch', fontsize=8)
        axes[r, c * 2 + 1].axis('off')
        axes[r, c * 2 + 2].imshow(msks[i], cmap='Reds')
        axes[r, c * 2 + 2].set_title('Lesion mask', fontsize=8)
        axes[r, c * 2 + 2].axis('off')

plt.suptitle('Sample Training Patches per Lesion Type (post-augmentation)', fontsize=14)
plt.tight_layout()
save_path = os.path.join(OUTPUT_BASE, 'sample_patches.png')
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'\nVisualization saved: {save_path}')
print('Preprocessing complete. Proceed to Notebook 2 (Segmentation Training + HSA).')